In [1]:
import os
import sys
from ultralytics import YOLO 
import yaml
import torch


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

In [2]:

data_cfg = {
    'path': config.YOLO_DATA_DIR,
    'train': "images/train",
    'val': "images/val",
    'nc': 1,         # number of classes
    'names': ['motor']  # class names
}

with open(f'{config.SRC}/data.yaml', 'w') as f:
    yaml.dump(data_cfg, f)

print('data.yaml written successfully')

data.yaml written successfully


In [ ]:
model = YOLO('yolov8x.pt')  
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT, name='motor_yolov8x_1280_long',
    epochs=300, patience=30,
    batch=8, imgsz=1280,
    optimizer='AdamW', lr0=1e-4, lrf=0.1, weight_decay=0.02,
    cos_lr=True, warmup_epochs=5,
    cache='disk', rect=True, multi_scale=False,
    augment=True,
    hsv_h=0.0, hsv_s=0.05, hsv_v=0.15,
    degrees=0.0, translate=0.05, scale=0.10, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.5,
    mosaic=0.0, mixup=0.0, cutmix=0.0,
    save_period=1
)


SyntaxError: '[31m[1mema[0m' is not a valid YOLO argument. 

    Arguments received: ['yolo', '--f=/run/user/2210974/jupyter/runtime/kernel-v3dd30392136aec4a08176d97a5fe8a7f73e2c124f.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['classify', 'segment', 'detect', 'pose', 'obb']
                MODE (required) is one of ['val', 'track', 'export', 'benchmark', 'train', 'predict']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo11n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo11n-seg.pt source='https://youtu.be/LNwODJXc

In [ ]:
model = YOLO('yolov8x.pt')
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT,
    name='motor_detection_optimized',
    epochs=50,
    batch=16,
    imgsz=640,
    multi_scale=False,
    rect=False,
    device = 'cuda',

    # --- Optimizer & LR ---
    optimizer='SGD',
    lr0=0.005,               # lower initial LR for longer learning
    lrf=0.05,                # decay to 5% of lr0 at end
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=5,         # longer warmup before decay

    # --- Early stopping & checkpointing ---
    patience=8,              # stop after 8 epochs without improvement
    save_period=1,
    verbose=True,

    # --- Loss weights (rebalanced) ---
    box=0.10,                # up-weight box regression
    cls=0.20,                # down-weight classification
    dfl=1.50,

    # --- Augmentations ---
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5,

    degrees=15,
    translate=0.1,
    scale=0.1,
    shear=2,
    fliplr=0.5,
    flipud=0.0,
    perspective=0.0,

    mosaic=0.5,
    mixup=0.1,
    cutmix=0.0,
)
print(results)


In [ ]:
model = YOLO('yolov8s.pt')
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT,
    name='medical_detection_optimized',
    epochs=50,
    batch=8,                 # smaller batch for higher-res medical scans
    imgsz=1024,              # larger input size to capture fine structures
    multi_scale=True,        # allow varying image scales each epoch
    rect=True,               # respect original aspect ratios

    # --- Optimizer & LR for small datasets ---
    optimizer='Adam',        # adaptive updates often help on limited data
    lr0=1e-3,                # start lower for stability
    lrf=0.1,                 # final LR = 10% of lr0
    weight_decay=1e-5,       # light regularization

    # --- Early stopping & checkpoints ---
    patience=15,             # give more epochs to improve on scarce positives
    save_period=1,
    verbose=True,

    # --- Rebalanced loss weights for small, subtle objects ---
    box=0.20,                # more emphasis on precise box placement
    cls=0.20,                # balanced class loss so detection isn’t overwhelmed
    dfl=2.50,                # sharpen edge localization for tiny structures

    # --- Augmentations tuned for medical imagery ---
    augment=True,
    hsv_h=0.0,               # no hue shift (often grayscale or consistent staining)
    hsv_s=0.0,
    hsv_v=0.1,               # slight brightness/contrast variation
    degrees=45,              # full rotations to handle arbitrary orientations
    translate=0.2,           # small shifts to simulate framing variation
    scale=0.2,               # zoom in/out for variable magnification
    shear=5,                 # minor shearing for acquisition distortions
    fliplr=0.5,              # horizontal flips
    flipud=0.5,              # vertical flips (if anatomical orientation is arbitrary)
    perspective=0.0001,      # minimal perspective warp

    mosaic=0.5,              # moderate mosaic to combine multiple fields of view
    mixup=0.0,               # disable mixup (can create unrealistic overlays)
    cutmix=0.0,              # disable cutmix (avoid unnatural tissue cuts)
)
print(results)


In [ ]:
# Load a model
last_weight   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/last.pt"

model = YOLO(last_weight)  # load a partially trained model

# Resume training
results = model.train(resume=True, epochs=12)


In [ ]:
# 5.1 Load model and run inference

# — adjust these paths to your layout —
weights_path   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/best.pt"
val_image_dir  = config.YOLO_VAL_IMAGE_DIR # your folder of val images
output_dir     = config.YOLO_OUTPUT_DIR    # where to save annotated images

# Load the best weights
model = YOLO(weights_path)

# Run prediction on the entire folder, save annotated images & TXT
results = model.predict(
    project=config.YOLO_RESULT_PREDICT,
    source=val_image_dir,
    imgsz=640,
    conf=0.10,
    iou=0.03,
    max_det=100,
    save=True,
    save_dir=output_dir,
    save_txt=True
)


print(f"✅ Predictions saved to {output_dir}")


In [ ]:
# 5.1 Load model and run inference

# — adjust these paths to your layout —
weights_path   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/best.pt"
val_image_dir  = config.YOLO_VAL_IMAGE_DIR # your folder of val images
output_dir     = config.YOLO_OUTPUT_DIR    # where to save annotated images

# Load the best weights
model = YOLO(weights_path)

# Run prediction on the entire folder, save annotated images & TXT
results = model.val(
    project=config.YOLO_RESULT_PREDICT,
    source=val_image_dir,
    imgsz=640,
    conf=0.22,
    iou=0.03,
    max_det=100,
    save=True,
    save_dir=output_dir,
    save_txt=True
)


print(f"✅ Predictions saved to {output_dir}")
